---
title: "2.2 Selecting rows and columns"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/02-selecting.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/02-data/02-selecting.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 35 minutes, including exercises · **Before you begin:** Complete [2.1 Loading data and looking at it](../02-data/01-loading-data.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Select columns by name and rows with a Boolean condition.
- Combine conditions with parentheses and use .loc.
- Sort a subset and keep its unit of observation clear.

## Why it matters

Before comparing income and life expectancy, decide which observations belong in the comparison. Taking the first 142 rows of a panel does not select one year. A **subset** is a selected portion of a table. We start with made-up numbers for fictional countries, then select one historical year from the running example.

## Worked example

### Choose columns

Import pandas and create a table you can check by hand. Every row is one fictional country-year. A single column name inside brackets selects a **Series**, pandas' one-column structure. A list of names inside brackets selects a DataFrame, even for a single selected column.

In [ ]:
import pandas as pd
small = pd.DataFrame({
    "country": ["Northland", "Northland", "Eastria", "Eastria"],
    "year": [2002, 2007, 2002, 2007],
    "life_exp": [70.0, 74.0, 76.0, 78.0],
    "gdp_pc": [12000, 15000, 25000, 30000],
})
small[["country", "life_exp"]]

The result has four rows and two columns. The inner list names what to keep; the outer brackets apply that selection to `small`. The original table still has its year and GDP columns.

### Test each row

Compare the year column with 2007 using `==`. This creates a **Boolean mask**, a series of True/False choices aligned with the rows. Use the mask with `.loc[rows, columns]` to select both observations and fields.

In [ ]:
is_2007 = small["year"] == 2007
print(is_2007.tolist())
small.loc[is_2007, ["country", "year", "life_exp"]]

The choices are False, True, False, True. The displayed result contains the two 2007 observations. The original row labels remain 1 and 3; filtering does not renumber them automatically.

### Combine conditions explicitly

`&` means both conditions must be True when comparing pandas columns. Put each comparison in parentheses. Here you want 2007 observations with life expectancy above 75 years, using an illustrative threshold.

In [ ]:
selected = small.loc[(small["year"] == 2007) & (small["life_exp"] > 75), ["country", "life_exp"]]
selected

Only fictional Eastria remains. Use `|` for either condition and `~` to reverse a Boolean mask. For one ordinary Python Boolean, `and` is appropriate; for whole pandas columns, use `&` with parentheses.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Select a cross-section from the panel

A **cross-section** contains observations from one time period. Load the panel, select 2007, and use `.copy()` to create a separate table for later changes. Sort life expectancy from largest to smallest and display five observations.

In [ ]:
countries = pd.read_csv(data_file)
year_2007 = countries.loc[countries["year"] == 2007, ["country", "region", "year", "life_exp", "gdp_pc", "population"]].copy()
ranked = year_2007.sort_values("life_exp", ascending=False)
print(year_2007.shape)
ranked[["country", "year", "life_exp"]].head()

There are 142 rows. Japan appears first by life expectancy in this extract for 2007. `ascending=False` requests descending order. A ranking describes these observations; it does not establish why countries differ. Equal values can tie, so a table's first row is not always a uniquely highest observation.

### Select multiple countries by membership

`.isin(...)` checks whether a value belongs to a supplied list. Here select Canada and Japan within the already filtered year.

In [ ]:
comparison = year_2007.loc[year_2007["country"].isin(["Canada", "Japan"]), ["country", "life_exp", "gdp_pc"]]
comparison

You see two country observations, with GDP per person and life expectancy measured in different units. You can compare levels, but not subtract GDP from life expectancy. Keep the source and units in your written interpretation.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Select another year

Create `year_2002` with all columns for 2002. Print the number of rows.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
year_2002 = countries.loc[countries["year"] == 2002].copy()
print(len(year_2002))

There are 142 rows. A year filter selects observations rather than arbitrary table positions.

</details>

### 2. Combine year and region

Select 2007 observations in Europe with .loc. Keep country, life expectancy and GDP per person. Store the result as `europe_2007`.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
europe_2007 = countries.loc[(countries["year"] == 2007) & (countries["region"] == "Europe"), ["country", "life_exp", "gdp_pc"]].copy()
print(europe_2007.shape)
europe_2007.head()

There are 30 countries in this extract's Europe group. This is a source-specific grouping, not a new geopolitical definition.

</details>

### 3. Inspect the lowest values

Sort the 2007 cross-section by life expectancy in ascending order and show three rows. What can the result establish?

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
lowest = year_2007.sort_values("life_exp")[["country", "life_exp"]].head(3)
lowest

Swaziland, Mozambique and Zambia have the lowest values in this historical extract for that year. Country names follow the old source. The ranking can motivate questions, but cannot diagnose causes or describe present conditions.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A column has the wrong capitalization

This intentionally asks for a missing column.

In [ ]:
try:
    countries["Life_Exp"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect `countries.columns`. The exact name is `life_exp`. Case and underscores matter.

### Python and is used on whole columns

Python tries to interpret each complete Series as one True/False answer. That is ambiguous.

In [ ]:
try:
    countries[(countries["year"] == 2007) and (countries["region"] == "Europe")]
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

The message says the truth value of a Series is ambiguous. For these column comparisons use `(condition_one) & (condition_two)`, then select rows with .loc.

## Recap

- Select by the meaning of a row, not just its position.
- Use == for comparison and a Boolean mask for filtering.
- Use .loc[rows, columns] and parenthesized & or | combinations.
- Sort explicitly; retain the observation year and units.

**Next:** [2.3 Creating and transforming variables](../02-data/03-transforming.ipynb).

> **Coming from Stata / R / Excel?**
>
> A year filter resembles Stata's `keep if year == 2007`, an R row filter or an Excel column filter. Assigning a pandas subset to a new name preserves the original DataFrame.